# jaxfne Colab Notebook

```python
import jaxfne as jtfne
```

**Version target:** v0.3.3+  
**Outputs:** figures, metrics, manifests  
**Proxy scope:** laminar_proxy_no_pde  


In [ ]:
# Install selector
INSTALL_MODE = "local"  # "local", "dev"
REPO_URL = "https://github.com/HNXJ/jaxfne.git"
BRANCH = "dev"

import sys
import subprocess

if 'google.colab' in sys.modules or INSTALL_MODE == "dev":
    if INSTALL_MODE == "dev":
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"git+{REPO_URL}@{BRANCH}"])

In [ ]:
# Canonical imports
import jaxfne as jtfne

import importlib
import json
from pathlib import Path

import jax
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Orientation gate
assert hasattr(jtfne, "SanityDeltaConfig"), "Missing SanityDeltaConfig"
assert importlib.import_module("jaxfne.sanity_runtime"), "Missing sanity_runtime module"
assert getattr(jtfne, "__version__", "") == "0.3.32-alpha", f"Invalid version: {jtfne.__version__}"

# Resolve paths relative to repo root
cwd = Path.cwd()
if cwd.name == "tutorials":
    repo_root = cwd.parent
else:
    repo_root = cwd

out_dir = repo_root / "outputs" / "v0333_colab_gemini_evidence"
out_dir.mkdir(parents=True, exist_ok=True)

orientation_report = {
    "jaxfne_version": jtfne.__version__,
    "jaxfne_file": jtfne.__file__,
    "has_sanity_delta": True,
    "has_sanity_runtime": True,
    "selected_path": "v032_delta_full_smoke",
    "truth_mode": "truth_safe_unverified",
    "claim_level": "computational_scaffold",
    "field_solver_status": "laminar_proxy_no_pde",
    "physical_amplitude_claim_allowed": False,
    "biological_learning_claim": False,
}

with open(out_dir / "orientation_report.json", "w") as f:
    json.dump(orientation_report, f, allow_nan=False, indent=2)

print("Saved orientation report.")
print(json.dumps(orientation_report, indent=2))

In [ ]:
# Task simulation
cfg = jtfne.SanityDeltaConfig.hierarchical_global_local_oddball(
    runtime_mode="full",
    seed=0,
    duration_ms=1000.0,
    dt_ms=0.1,
)
paradigm = cfg.make_paradigm()
model = cfg.construct()
gate = paradigm.make_fixation_gate()
model = model.enable_plasticity()
backup = model.initialize_backup(paradigm=paradigm, history_ms=1000.0)

episode = model.run_task(
    paradigm=paradigm,
    gate=gate,
    backup=backup,
    runtime_mode="full",
)
episode = episode.probe(
    readouts=("spk", "vm", "source", "lfp_proxy", "csd_proxy", "eeg_proxy", "meg_proxy")
)
validation = episode.validate(
    checks=(
        "finite_outputs",
        "strict_json",
        "backup_resume_equivalence",
        "proxy_safe_readout_names",
        "truth_gates_preserved",
    )
)
episode.export(
    artifact_dir=str(out_dir / "reports"),
    strict_json=True,
)
print("Task completed and reports exported.")

In [ ]:
# Finite checks
vm = episode.signals.get("vm")
spk = episode.signals.get("spk")
source = episode.signals.get("source")
lfp_proxy = episode.signals.get("lfp_proxy")
csd_proxy = episode.signals.get("csd_proxy")
eeg_proxy = episode.signals.get("eeg_proxy")
meg_proxy = episode.signals.get("meg_proxy")

assert np.all(np.isfinite(vm)), "vm contains non-finite values"
assert np.all(np.isfinite(spk)), "spk contains non-finite values"
assert np.all(np.isfinite(source)), "source contains non-finite values"
assert np.all(np.isfinite(lfp_proxy)), "lfp_proxy contains non-finite values"
assert np.all(np.isfinite(csd_proxy)), "csd_proxy contains non-finite values"
assert np.all(np.isfinite(eeg_proxy)), "eeg_proxy contains non-finite values"
assert np.all(np.isfinite(meg_proxy)), "meg_proxy contains non-finite values"

spike_count = float(np.sum(spk))
assert spike_count > 0, f"Expected spikes > 0, got {spike_count}"

for check_name, check_val in validation.items():
    assert check_val is True, f"Validation check {check_name} failed"

print("All finite checks passed. Spike count:", spike_count)

In [ ]:
# Figure generation
fig_dir = out_dir / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

# 1. raster.png
plt.figure(figsize=(10, 5))
times = np.arange(spk.shape[0]) * 0.1
for neuron_idx in range(0, spk.shape[1], 10):
    spike_times = times[spk[:, neuron_idx] > 0]
    plt.scatter(spike_times, [neuron_idx] * len(spike_times), color="black", s=2)
plt.title("simulated spikes")
plt.xlabel("Time (ms)")
plt.ylabel("Neuron Index")
plt.tight_layout()
plt.savefig(fig_dir / "raster.png", dpi=100)
plt.close()

# 2. mean_vm_by_area.png
plt.figure(figsize=(10, 5))
areas_names = ["V1a", "V1b", "V4", "MT", "PFC"]
for i, area in enumerate(areas_names):
    start, end = i * 100, (i + 1) * 100
    mean_vm = np.mean(vm[:, start:end], axis=1)
    plt.plot(times, mean_vm, label=area)
plt.title("Mean Membrane Potential by Area")
plt.xlabel("Time (ms)")
plt.ylabel("Voltage (mV)")
plt.legend()
plt.tight_layout()
plt.savefig(fig_dir / "mean_vm_by_area.png", dpi=100)
plt.close()

# 3. proxy_readout_summary.png
plt.figure(figsize=(10, 5))
plt.plot(times, np.mean(np.abs(lfp_proxy), axis=1), label="lfp_proxy")
plt.plot(times, np.mean(np.abs(csd_proxy), axis=1), label="csd_proxy")
plt.plot(times, np.mean(np.abs(eeg_proxy), axis=1), label="eeg_proxy")
plt.plot(times, np.mean(np.abs(meg_proxy), axis=1), label="meg_proxy")
plt.title("Proxy Readouts Mean Absolute Activity")
plt.xlabel("Time (ms)")
plt.ylabel("Amplitude")
plt.legend()
plt.tight_layout()
plt.savefig(fig_dir / "proxy_readout_summary.png", dpi=100)
plt.close()

print("Figures generated.")

In [ ]:
# Evidence manifest
evidence_manifest = {
    "jaxfne_version": jtfne.__version__,
    "jaxfne_file": jtfne.__file__,
    "runtime_mode": "full",
    "duration_ms": 1000.0,
    "dt_ms": 0.1,
    "seed": 0,
    "areas": ["V1a", "V1b", "V4", "MT", "PFC"],
    "n_neurons": 500,
    "vm_shape": list(vm.shape),
    "spk_shape": list(spk.shape),
    "source_shape": list(source.shape),
    "lfp_proxy_shape": list(lfp_proxy.shape),
    "csd_proxy_shape": list(csd_proxy.shape),
    "eeg_proxy_shape": list(eeg_proxy.shape),
    "meg_proxy_shape": list(meg_proxy.shape),
    "spike_count": spike_count,
    "finite_outputs": True,
    "validation": validation,
    "generated_reports": [
        "orientation_report.json",
        "reports/manifest.json",
        "reports/validation_report.json",
        "reports/task_schedule.json",
        "reports/backup_resume_report.json",
        "reports/probe_report.json",
        "reports/plasticity_report.json"
    ],
    "generated_figures": [
        "figures/raster.png",
        "figures/mean_vm_by_area.png",
        "figures/proxy_readout_summary.png"
    ],
    "area_indexing_method": "five_contiguous_100_neuron_blocks",
    "truth_mode": "truth_safe_unverified",
    "claim_level": "computational_scaffold",
    "field_solver_status": "laminar_proxy_no_pde",
    "physical_amplitude_claim_allowed": False,
    "biological_learning_claim": False,
    "interpretation": "This notebook verifies Colab/Gemini execution of the jaxfne dev SanityDelta full-runtime path and produces finite simulated/proxy readouts plus strict JSON reports.",
    "limits": "These outputs are computational scaffold readouts and proxy readouts, not calibrated physical measurements or biological validation."
}

with open(out_dir / "evidence_manifest.json", "w") as f:
    json.dump(evidence_manifest, f, allow_nan=False, indent=2)

print("Saved evidence manifest.")